In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/recruit-restaurant-visitor-forecasting/air_reserve.csv.zip
/kaggle/input/competitions/recruit-restaurant-visitor-forecasting/hpg_store_info.csv.zip
/kaggle/input/competitions/recruit-restaurant-visitor-forecasting/hpg_reserve.csv.zip
/kaggle/input/competitions/recruit-restaurant-visitor-forecasting/sample_submission.csv.zip
/kaggle/input/competitions/recruit-restaurant-visitor-forecasting/air_visit_data.csv.zip
/kaggle/input/competitions/recruit-restaurant-visitor-forecasting/air_store_info.csv.zip
/kaggle/input/competitions/recruit-restaurant-visitor-forecasting/date_info.csv.zip
/kaggle/input/competitions/recruit-restaurant-visitor-forecasting/store_id_relation.csv.zip


In [2]:
#ライブラリの読み込み
import numpy as np
import pandas as pd
import gc
import pickle
import os
import datetime as dt

# plot
import matplotlib.pyplot as plt

# LightGBM
import lightgbm as lgb

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_squared_log_error

import warnings
warnings.simplefilter("ignore")

# 表示桁数の指定
pd.options.display.float_format = '{:10.4f}'.format

In [3]:
#データの読み込み

base = "/kaggle/input/competitions/recruit-restaurant-visitor-forecasting/"

air_visit = pd.read_csv(base + "air_visit_data.csv.zip")
air_store = pd.read_csv(base + "air_store_info.csv.zip")
hpg_store = pd.read_csv(base + "hpg_store_info.csv.zip")
air_reserve = pd.read_csv(base + "air_reserve.csv.zip")
hpg_reserve = pd.read_csv(base + "hpg_reserve.csv.zip")
store_relation = pd.read_csv(base + "store_id_relation.csv.zip")
date_info = pd.read_csv(base + "date_info.csv.zip")

In [4]:
#データの確認
dfs = {
    "air_visit": air_visit,
    "air_store": air_store,
    "hpg_store": hpg_store,
    "air_reserve": air_reserve,
    "hpg_reserve": hpg_reserve,
    "store_relation": store_relation,
    "date_info": date_info
}

for name, df_ in dfs.items():
    print("="*40)
    print(name)
    print("shape:", df_.shape)
    display(df_.head(3))
    print(df_.info())

air_visit
shape: (252108, 3)


,air_store_id,visit_date,visitors
0,air_ba937bf13d40fb24,2016-01-13,25
1,air_ba937bf13d40fb24,2016-01-14,32
2,air_ba937bf13d40fb24,2016-01-15,29


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 252108 entries, 0 to 252107
Data columns (total 3 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   air_store_id  252108 non-null  object
 1   visit_date    252108 non-null  object
 2   visitors      252108 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 5.8+ MB
None
air_store
shape: (829, 5)


,air_store_id,air_genre_name,air_area_name,latitude,longitude
0,air_0f0cdeee6c9bf3d7,Italian/French,Hyōgo-ken Kōbe-shi Kumoidōri,34.6951,135.1979
1,air_7cc17a324ae5c7dc,Italian/French,Hyōgo-ken Kōbe-shi Kumoidōri,34.6951,135.1979
2,air_fee8dcf4d619598e,Italian/French,Hyōgo-ken Kōbe-shi Kumoidōri,34.6951,135.1979


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 829 entries, 0 to 828
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   air_store_id    829 non-null    object 
 1   air_genre_name  829 non-null    object 
 2   air_area_name   829 non-null    object 
 3   latitude        829 non-null    float64
 4   longitude       829 non-null    float64
dtypes: float64(2), object(3)
memory usage: 32.5+ KB
None
hpg_store
shape: (4690, 5)


...

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000320 entries, 0 to 2000319
Data columns (total 4 columns):
 #   Column            Dtype 
---  ------            ----- 
 0   hpg_store_id      object
 1   visit_datetime    object
 2   reserve_datetime  object
 3   reserve_visitors  int64 
dtypes: int64(1), object(3)
memory usage: 61.0+ MB
None
store_relation
shape: (150, 2)


,air_store_id,hpg_store_id
0,air_63b13c56b7201bd9,hpg_4bc649e72e2a239a
1,air_a24bf50c3e90d583,hpg_c34b496d0305a809
2,air_c7f78b4f3cba33ff,hpg_cd8ae0d9bbd58ff9


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   air_store_id  150 non-null    object
 1   hpg_store_id  150 non-null    object
dtypes: object(2)
memory usage: 2.5+ KB
None
date_info
shape: (517, 3)


,calendar_date,day_of_week,holiday_flg
0,2016-01-01,Friday,1
1,2016-01-02,Saturday,1
2,2016-01-03,Sunday,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   calendar_date  517 non-null    object
 1   day_of_week    517 non-null    object
 2   holiday_flg    517 non-null    int64 
dtypes: int64(1), object(2)
memory usage: 12.2+ KB
None


In [5]:
# train と test を作る
# train は air_visit
train = air_visit.copy()
train["visit_date"] = pd.to_datetime(train["visit_date"])

# test を作る
stores = air_store["air_store_id"].unique()
test_dates = pd.date_range("2017-04-23", "2017-05-31")

test = pd.MultiIndex.from_product(
    [stores, test_dates],
    names=["air_store_id", "visit_date"]
).to_frame(index=False)

test["id"] = test["air_store_id"] + "_" + test["visit_date"].dt.strftime("%Y-%m-%d")
test["visitors"] = np.nan

In [6]:
#train と test を縦に結合する
df_all = pd.concat([train, test], ignore_index=True)

In [7]:
#visit_date を “純粋な日付” に統一する
df_all["visit_date"] = pd.to_datetime(df_all["visit_date"].dt.date)

In [8]:
#air_reserve の前処理
# 日付を datetime に変換
air_reserve["visit_datetime"] = pd.to_datetime(air_reserve["visit_datetime"])
air_reserve["reserve_datetime"] = pd.to_datetime(air_reserve["reserve_datetime"])

# visit_date（来店日）を date → datetime に統一
air_reserve["visit_date"] = air_reserve["visit_datetime"].dt.date
air_reserve["visit_date"] = pd.to_datetime(air_reserve["visit_date"])

# 予約から来店までの日数
air_reserve["reserve_lead_days"] = (
    air_reserve["visit_datetime"] - air_reserve["reserve_datetime"]
).dt.days

In [9]:
#hpg_reserve の前処理
# 日付を datetime に変換
hpg_reserve["visit_datetime"] = pd.to_datetime(hpg_reserve["visit_datetime"])
hpg_reserve["reserve_datetime"] = pd.to_datetime(hpg_reserve["reserve_datetime"])

# visit_date（来店日）を date → datetime に統一
hpg_reserve["visit_date"] = hpg_reserve["visit_datetime"].dt.date
hpg_reserve["visit_date"] = pd.to_datetime(hpg_reserve["visit_date"])

# 予約から来店までの日数
hpg_reserve["reserve_lead_days"] = (
    hpg_reserve["visit_datetime"] - hpg_reserve["reserve_datetime"]
).dt.days

In [10]:
#hpg → air_store_id に変換
hpg_reserve = hpg_reserve.merge(store_relation, on="hpg_store_id", how="left")

# air_store_id が無い行は使えないので除外
hpg_reserve = hpg_reserve.dropna(subset=["air_store_id"])

In [11]:
#air_reserve と hpg_reserve を縦に結合して reserve_all を作る
reserve_all = pd.concat([
    air_reserve[["air_store_id", "visit_date", "reserve_visitors", "reserve_lead_days"]],
    hpg_reserve[["air_store_id", "visit_date", "reserve_visitors", "reserve_lead_days"]]
], axis=0)

In [12]:
#予約特徴量 reserve_agg を作る
reserve_agg = (
    reserve_all.groupby(["air_store_id", "visit_date"])
               .agg({
                   "reserve_visitors": ["sum", "mean", "count"],
                   "reserve_lead_days": ["mean"]
               })
               .reset_index()
)

reserve_agg.columns = [
    "air_store_id", "visit_date",
    "res_sum", "res_mean", "res_count", "res_lead_mean"
]

In [13]:
#visit_date を df_all と同じ型に揃える
reserve_agg["visit_date"] = pd.to_datetime(reserve_agg["visit_date"])

In [14]:
#df_all に予約特徴量を 1 回だけ merge する
df_all = df_all.merge(
    reserve_agg,
    on=["air_store_id", "visit_date"],
    how="left"
)

df_all[["res_sum", "res_mean", "res_count", "res_lead_mean"]] = \
    df_all[["res_sum", "res_mean", "res_count", "res_lead_mean"]].fillna(0)

In [15]:
#air_store（店舗情報）を df_all に merge
df_all = df_all.merge(air_store, on="air_store_id", how="left")

In [16]:
#date_info（曜日・休日情報）を df_all に merge
#date_info の日付を datetime に変換
date_info["calendar_date"] = pd.to_datetime(date_info["calendar_date"])

#列名を visit_date に揃える
date_info = date_info.rename(columns={"calendar_date": "visit_date"})

#df_all と date_info を visit_date で結合
df_all = df_all.merge(date_info, on="visit_date", how="left")

In [17]:
#データの確認
print(df_all.shape)
display(df_all.head())
df_all.info()
df_all.describe()

(284439, 14)


,air_store_id,visit_date,visitors,id,res_sum,res_mean,res_count,res_lead_mean,air_genre_name,air_area_name,latitude,longitude,day_of_week,holiday_flg
0,air_ba937bf13d40fb24,2016-01-13,25.0000,NaN,0.0000,0.0000,0.0000,0.0000,Dining bar,Tōkyō-to Minato-ku Shibakōen,35.6581,139.7516,Wednesday,0
1,air_ba937bf13d40fb24,2016-01-14,32.0000,NaN,0.0000,0.0000,0.0000,0.0000,Dining bar,Tōkyō-to Minato-ku Shibakōen,35.6581,139.7516,Thursday,0
2,air_ba937bf13d40fb24,2016-01-15,29.0000,NaN,0.0000,0.0000,0.0000,0.0000,Dining bar,Tōkyō-to Minato-ku Shibakōen,35.6581,139.7516,Friday,0
3,air_ba937bf13d40fb24,2016-01-16,22.0000,NaN,0.0000,0.0000,0.0000,0.0000,Dining bar,Tōkyō-to Minato-ku Shibakōen,35.6581,139.7516,Saturday,0
4,air_ba937bf13d40fb24,2016-01-18,6.0000,NaN,0.0000,0.0000,0.0000,0.0000,Dining bar,Tōkyō-to Minato-ku Shibakōen,35.6581,139.7516,Monday,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284439 entries, 0 to 284438
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   air_store_id    284439 non-null  object        
 1   visit_date      284439 non-null  datetime64[ns]
 2   visitors        252108 non-null  float64       
 3   id              32331 non-null   object        
 4   res_sum         284439 non-null  float64       
 5   res_mean        284439 non-null  float64       
 6   res_count       284439 non-null  float64       
 7   res_lead_mean   284439 non-null  float64       
 8   air_genre_name  284439 non-null  object        
 9   air_area_name   284439 non-null  object        
 10  latitude        284439 non-null  float64       
 11  longitude       284439 non-null  float64       
 12  day_of_week     284439 non-null  object        
 13  holiday_flg     284439 non-null  int64         
dtypes: datetime64[ns](1), float64(7), in

,visit_date,visitors,res_sum,res_mean,res_count,res_lead_mean,latitude,longitude,holiday_flg
count,284439,252108.0000,284439.0000,284439.0000,284439.0000,284439.0000,284439.0000,284439.0000,284439.0000
mean,2016-11-05 13:41:34.101019904,20.9738,1.7296,0.6487,0.3890,0.7846,35.6170,137.3644,0.0566
min,2016-01-01 00:00:00,1.0000,0.0000,0.0000,0.0000,0.0000,33.2120,130.1956,0.0000
25%,2016-08-03 00:00:00,9.0000,0.0000,0.0000,0.0000,0.0000,34.6923,135.3416,0.0000
50%,2016-11-15 00:00:00,17.0000,0.0000,0.0000,0.0000,0.0000,35.6581,139.6720,0.0000
75%,2017-02-28 00:00:00,29.0000,0.0000,0.0000,0.0000,0.0000,35.6940,139.7516,0.0000
max,2017-05-31 00:00:00,877.0000,1633.0000,100.0000,305.0000,350.7745,44.0206,144.2734,1.0000
std,NaN,16.7570,7.6363,2.5176,1.5536,3.7991,2.0490,3.6690,0.2310


In [18]:
#特徴量追加
#移動平均・移動標準偏差（rolling features）
df_all = df_all.sort_values(["air_store_id", "visit_date"])

df_all["visitors_7d_mean"] = df_all.groupby("air_store_id")["visitors"].transform(lambda x: x.rolling(7).mean())
df_all["visitors_14d_mean"] = df_all.groupby("air_store_id")["visitors"].transform(lambda x: x.rolling(14).mean())
df_all["visitors_28d_mean"] = df_all.groupby("air_store_id")["visitors"].transform(lambda x: x.rolling(28).mean())

df_all["visitors_7d_std"] = df_all.groupby("air_store_id")["visitors"].transform(lambda x: x.rolling(7).std())
df_all["visitors_14d_std"] = df_all.groupby("air_store_id")["visitors"].transform(lambda x: x.rolling(14).std())

df_all = df_all.reset_index(drop=True)

In [19]:
#季節性（year, month, week_of_year）
df_all["year"] = df_all["visit_date"].dt.year
df_all["month"] = df_all["visit_date"].dt.month
df_all["week_of_year"] = df_all["visit_date"].dt.isocalendar().week.astype(int)

In [20]:
#祝日フラグ（before / after）
df_all["before_holiday_flg"] = df_all["holiday_flg"].shift(-1).fillna(0).astype(int)
df_all["after_holiday_flg"] = df_all["holiday_flg"].shift(1).fillna(0).astype(int)

In [21]:
#店舗ごとの長期平均（store_mean_visitors）
df_all["store_mean_visitors"] = df_all.groupby("air_store_id")["visitors"].transform("mean")

In [22]:
import numpy as np
from sklearn.linear_model import LinearRegression

#店舗ごとに時系列順に並べて、rolling window 内で線形回帰して傾きを取る
def add_slope_features(df, group_col="air_store_id", target_col="visitors"):
    df = df.sort_values(["air_store_id", "visit_date"]).copy()

    def calc_slope(x):
        # NaN を除外
        x = x.dropna()
        if len(x) < 2:
            return np.nan
        lr = LinearRegression()
        lr.fit(np.arange(len(x)).reshape(-1, 1), x)
        return lr.coef_[0]

    windows = [7, 14, 28]
    for w in windows:
        df[f"slope_{w}d"] = (
            df.groupby(group_col)[target_col]
              .rolling(window=w, min_periods=2)
              .apply(calc_slope, raw=False)
              .reset_index(level=0, drop=True)
        )

    return df
    

In [23]:
#df_all に slope（傾き）特徴量を追加する
df_all = add_slope_features(df_all)

In [24]:
#① train/test に分割（df_all → train_df/test_df）
train_df = df_all[df_all["visitors"].notnull()].copy()
test_df  = df_all[df_all["visitors"].isnull()].copy()

#ここで必ず index をリセットする
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

In [25]:
#② CV 用の df を作る（df_all を汚さない）
df_cv = train_df.copy()

In [26]:
#③ list_cv_date を作る（7日×4fold）
list_cv_date = [
    [
        df_cv[df_cv.visit_date < "2017-04-16"].visit_date.unique(),
        df_cv[(df_cv.visit_date >= "2017-04-16") & (df_cv.visit_date <= "2017-04-22")].visit_date.unique(),
    ],
    [
        df_cv[df_cv.visit_date < "2017-04-09"].visit_date.unique(),
        df_cv[(df_cv.visit_date >= "2017-04-09") & (df_cv.visit_date <= "2017-04-15")].visit_date.unique(),
    ],
    [
        df_cv[df_cv.visit_date < "2017-04-02"].visit_date.unique(),
        df_cv[(df_cv.visit_date >= "2017-04-02") & (df_cv.visit_date <= "2017-04-08")].visit_date.unique(),
    ],
    [
        df_cv[df_cv.visit_date < "2017-03-26"].visit_date.unique(),
        df_cv[(df_cv.visit_date >= "2017-03-26") & (df_cv.visit_date <= "2017-04-01")].visit_date.unique(),
    ],
]

In [27]:
#④ 日付ベース CV を作る関数（make_date_folds）
def make_date_folds(df, list_cv_date):
    """
    df: input_x と同じ行数で、visit_date を含んでいる DataFrame
    list_cv_date: [
        [train_dates_array, valid_dates_array],
        ...
    ]
    """
    cv = []
    for train_dates, valid_dates in list_cv_date:
        idx_tr = df[df["visit_date"].isin(train_dates)].index.values
        idx_va = df[df["visit_date"].isin(valid_dates)].index.values
        cv.append((idx_tr, idx_va))
    return cv

In [28]:
#⑤ 学習データセット作成（x_train, y_train, id_train, date_train）
x_train = train_df.drop(columns=["visitors", "air_store_id", "visit_date"])
y_train = train_df["visitors"]
id_train = train_df[["air_store_id"]]
date_train = train_df["visit_date"]

# index を完全に揃える（これが決定打）
x_train = x_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
id_train = id_train.reset_index(drop=True)
date_train = date_train.reset_index(drop=True)

In [29]:
#⑥ カテゴリ変換（LightGBM 用）
for col in x_train.columns:
    if x_train[col].dtype == "O":
        x_train[col] = x_train[col].astype("category")

In [30]:
#学習関数の定義
def train_lgb(input_x,
              input_y,
              input_id,
              input_date,   # ← ここを追加（Series: visit_date）
              params,
              list_cv_date,
              list_nfold=None,
             ):
    
    if list_nfold is None:
        list_nfold = list(range(len(list_cv_date)))
    
    train_oof = np.full(len(input_x), np.nan)
    metrics = []
    imp = pd.DataFrame()

    # CV 用に date をくっつけた DF を作る
    df_cv = input_x.copy()
    df_cv["visit_date"] = input_date.values

    # 日付ベースの fold index を作成
    cv = make_date_folds(df_cv, list_cv_date)

    for nfold in list_nfold:
        print("-"*20, f"fold {nfold}", "-"*20)

        idx_tr, idx_va = cv[nfold]
        x_tr, y_tr, id_tr = input_x.loc[idx_tr, :], input_y[idx_tr], input_id.loc[idx_tr, :]
        x_va, y_va, id_va = input_x.loc[idx_va, :], input_y[idx_va], input_id.loc[idx_va, :]

        print(x_tr.shape, y_tr.shape, id_tr.shape)
        print(x_va.shape, y_va.shape, id_va.shape)

        model = lgb.LGBMRegressor(**params)
        model.fit(
            x_tr,
            y_tr,
            eval_set=[(x_tr, y_tr), (x_va, y_va)],
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, verbose=True),
                lgb.log_evaluation(100),
            ],
        )

        # モデル保存
        fname_lgb = f"model_lgb_fold{nfold}.pickle"
        with open(fname_lgb, "wb") as f:
            pickle.dump(model, f, protocol=4)

        # 予測
        y_tr_pred = model.predict(x_tr)
        y_va_pred = model.predict(x_va)

        # ★ここでクリップ（RMSLE のため）
        y_tr_pred = np.clip(y_tr_pred, 0, None)
        y_va_pred = np.clip(y_va_pred, 0, None)

        # RMSLE
        rmsle_tr = np.sqrt(mean_squared_log_error(y_tr, y_tr_pred))
        rmsle_va = np.sqrt(mean_squared_log_error(y_va, y_va_pred))

        metrics.append([nfold, rmsle_tr, rmsle_va])

        print("[RMSLE] tr:{:.4f}, va:{:.4f}".format(rmsle_tr, rmsle_va))

        # OOF
        train_oof[idx_va] = y_va_pred

        # importance
        _imp = pd.DataFrame({
            "col": input_x.columns,
            "imp": model.feature_importances_,
            "nfold": nfold
        })
        imp = pd.concat([imp, _imp], axis=0)

    # --- fold 終了後 ---
    print("-"*20, "result", "-"*20)
    metrics = np.array(metrics)
    print(metrics)

    print("[cv] rmsle_tr:{:.4f}+-{:.4f}, rmsle_va:{:.4f}+-{:.4f}".format(
        metrics[:,1].mean(), metrics[:,1].std(),
        metrics[:,2].mean(), metrics[:,2].std(),
    ))

    # OOF RMSLE
    # OOF が埋まっている行だけ使う（時系列CVでは必須）
    mask = ~np.isnan(train_oof)


    rmsle_oof = np.sqrt(mean_squared_log_error(input_y[mask], train_oof[mask]))
    print("[oof] RMSLE:{:.4f}".format(rmsle_oof))
    

    # OOF 出力
    train_oof = pd.concat([
        input_id.reset_index(drop=True),
        pd.DataFrame({"pred": train_oof})
    ], axis=1)

    # importance 集計
    imp = imp.groupby("col")["imp"].agg(["mean", "std"]).reset_index(drop=False)
    imp.columns = ["col", "imp", "imp_std"]

    return train_oof, imp, metrics

In [31]:
#パラメータを設定
params = {
    'boosting_type': 'gbdt',
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 32,
    'n_estimators': 100000,
    "random_state": 123,
    "importance_type": "gain",
}

In [32]:
#学習の実行
train_oof, imp, metrics = train_lgb(
    x_train,
    y_train,
    id_train,
    date_train,
    params,
    list_cv_date=list_cv_date,
    list_nfold=[0,1,2,3],  # 4fold の場合
)

-------------------- fold 0 --------------------
(247096, 25) (247096,) (247096, 1)
(5012, 25) (5012,) (5012, 1)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.021710 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3331
[LightGBM] [Info] Number of data points in the train set: 247096, number of used features: 24
[LightGBM] [Info] Start training from score 20.960473
Training until validation scores don't improve for 50 rounds
[100]	training's rmse: 8.13318	valid_1's rmse: 11.7752
[200]	training's rmse: 7.61501	valid_1's rmse: 11.4426
Early stopping, best iteration is:
[207]	training's rmse: 7.58891	valid_1's rmse: 11.4229
[RMSLE] tr:0.4276, va:0.4260
-------------------- fold 1 --------------------
(242100, 25) (242100,) (242100, 1)
(4996, 25) (4996,) (4996, 1)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of te

In [33]:
#説明変数の重要度の確認
imp.sort_values("imp", ascending=False)[:10]

,col,imp,imp_std
21,visitors_7d_mean,351419801.3152,11218911.5831
16,slope_7d,105188866.4858,2159960.6922
4,day_of_week,31101121.8415,933921.9087
22,visitors_7d_std,17597684.9668,1719545.1749
1,air_area_name,16836450.3309,2119813.7825
17,store_mean_visitors,8883283.3301,1201030.5247
13,res_sum,6757981.3290,179262.8974
2,air_genre_name,3827210.9137,605732.9182
20,visitors_28d_mean,2424827.6194,569331.8025
18,visitors_14d_mean,2153350.0793,826004.1350


In [34]:
#提出用コード
# ===========================
# 1. test 用データの準備
# ===========================

# test_df は df_all から作っている前提
# test 用特徴量（train と同じカラムを使う）
x_test = test_df.drop(columns=["air_store_id", "visit_date", "visitors"])

# カテゴリ変換（train と同じ処理）
for col in x_test.columns:
    if x_test[col].dtype == "O":
        x_test[col] = x_test[col].astype("category")

# ===========================
# 2. fold モデルを読み込んで予測
# ===========================

preds = np.zeros(len(x_test))

for nfold in range(len(list_cv_date)):
    fname_lgb = f"model_lgb_fold{nfold}.pickle"
    with open(fname_lgb, "rb") as f:
        model = pickle.load(f)

    pred_fold = model.predict(x_test)
    pred_fold = np.clip(pred_fold, 0, None)  # RMSLE 用にクリップ
    preds += pred_fold / len(list_cv_date)

# ===========================
# 3. submission の作成
# ===========================

submission = pd.DataFrame({
    "id": test_df["air_store_id"] + "_" + test_df["visit_date"].astype(str),
    "visitors": preds
})

# ===========================
# 4. CSV 出力
# ===========================

submission.to_csv("submission.csv", index=False)

print("submission.csv を作成しました")
submission.head()

submission.csv を作成しました


,id,visitors
0,air_00a91d42b08b08d9_2017-04-23,21.6517
1,air_00a91d42b08b08d9_2017-04-24,15.8047
2,air_00a91d42b08b08d9_2017-04-25,20.2106
3,air_00a91d42b08b08d9_2017-04-26,13.2945
4,air_00a91d42b08b08d9_2017-04-27,11.9516
